# 🗺️ Interactive Route Visualization Feature

This cell adds interactive maps to visualize the travel routes between cities. Maps help users understand the geographical context of their journey and compare different route options visually.

In [2]:
import folium
from folium import plugins
import requests

def show_route_map(source, destination):
    """Display an interactive map showing the route between source and destination"""
    # Get city coordinates from our recommender
    city_coords = {
        'Delhi': (28.6139, 77.2090),
        'Mumbai': (19.0760, 72.8777),
        'Bangalore': (12.9716, 77.5946),
        'Kolkata': (22.5726, 88.3639),
        'Chennai': (13.0827, 80.2707),
        'Hyderabad': (17.3850, 78.4867),
        'Pune': (18.5204, 73.8567),
        'Agra': (27.1767, 78.0081),
        # Add coordinates for other cities as needed
    }
    
    # Create map centered between the two cities
    if source in city_coords and destination in city_coords:
        src_lat, src_lon = city_coords[source]
        dst_lat, dst_lon = city_coords[destination]
        center_lat = (src_lat + dst_lat) / 2
        center_lon = (src_lon + dst_lon) / 2
        
        # Create the map
        m = folium.Map(location=[center_lat, center_lon], zoom_start=5)
        
        # Add markers for source and destination
        folium.Marker(
            [src_lat, src_lon],
            popup=source,
            tooltip=source,
            icon=folium.Icon(color="green", icon="play")
        ).add_to(m)
        
        folium.Marker(
            [dst_lat, dst_lon],
            popup=destination,
            tooltip=destination,
            icon=folium.Icon(color="red", icon="flag")
        ).add_to(m)
        
        # Add line connecting the cities - create different lines for different transport modes
        folium.PolyLine(
            [(src_lat, src_lon), (dst_lat, dst_lon)],
            color='blue',
            weight=3,
            opacity=0.7,
            dash_array='10',
            tooltip="Flight Route"
        ).add_to(m)
        
        # Create a more meandering route for train
        train_waypoint1 = (src_lat + (dst_lat - src_lat) * 0.33 + 0.5, src_lon + (dst_lon - src_lon) * 0.33 - 0.5)
        train_waypoint2 = (src_lat + (dst_lat - src_lat) * 0.66 - 0.5, src_lon + (dst_lon - src_lon) * 0.66 + 0.5)
        train_route = [(src_lat, src_lon), train_waypoint1, train_waypoint2, (dst_lat, dst_lon)]
        
        folium.PolyLine(
            train_route,
            color='red',
            weight=3,
            opacity=0.7,
            tooltip="Train Route"
        ).add_to(m)
        
        # Create a different route for bus
        bus_waypoint1 = (src_lat + (dst_lat - src_lat) * 0.4 - 0.3, src_lon + (dst_lon - src_lon) * 0.4 + 0.3)
        bus_waypoint2 = (src_lat + (dst_lat - src_lat) * 0.7 + 0.3, src_lon + (dst_lon - src_lon) * 0.7 - 0.3)
        bus_route = [(src_lat, src_lon), bus_waypoint1, bus_waypoint2, (dst_lat, dst_lon)]
        
        folium.PolyLine(
            bus_route,
            color='green',
            weight=3,
            opacity=0.7,
            tooltip="Bus Route"
        ).add_to(m)
        
        # Add distance measurement
        plugins.MeasureControl(position='topright', primary_length_unit='kilometers').add_to(m)
        
        # Add legend
        legend_html = '''
        <div style="position: fixed; bottom: 50px; left: 50px; 
                    border: 2px solid grey; z-index: 9999; font-size: 14px;
                    background-color: white; padding: 10px; border-radius: 5px;">
          <p><span style="color: blue; font-weight: bold;">———</span> Flight Route</p>
          <p><span style="color: red; font-weight: bold;">———</span> Train Route</p>
          <p><span style="color: green; font-weight: bold;">———</span> Bus Route</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legend_html))
        
        return m
    else:
        return "Coordinates not available for selected cities."

# Example: This will be called from our recommendation function
# show_route_map("Delhi", "Mumbai")

ModuleNotFoundError: No module named 'folium'

# 🔍 Advanced Filtering & 📊 Price Analytics Features

These cells add advanced filtering options for transportation choices and price trend analysis to help users make more informed decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def create_filter_widgets():
    """Create widgets for advanced filtering of transportation options"""
    transport_type = widgets.SelectMultiple(
        options=['All', 'Bus', 'Train', 'Flight'],
        value=['All'],
        description='Transport:',
        disabled=False,
        layout=widgets.Layout(width='200px')
    )
    
    price_range = widgets.IntRangeSlider(
        value=[500, 5000],
        min=0,
        max=10000,
        step=100,
        description='Price Range:',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        layout=widgets.Layout(width='300px')
    )
    
    departure_time = widgets.SelectMultiple(
        options=['All', 'Early Morning (4-8 AM)', 'Morning (8-12 PM)', 'Afternoon (12-4 PM)', 'Evening (4-8 PM)', 'Night (8 PM-12 AM)', 'Late Night (12-4 AM)'],
        value=['All'],
        description='Departure:',
        disabled=False,
        layout=widgets.Layout(width='250px')
    )
    
    comfort_level = widgets.SelectMultiple(
        options=['All', 'Budget', 'Standard', 'Premium'],
        value=['All'],
        description='Comfort:',
        disabled=False,
        layout=widgets.Layout(width='200px')
    )
    
    sort_by = widgets.Dropdown(
        options=['Price (Low to High)', 'Duration (Shortest)', 'Departure (Earliest)', 'Rating (Highest)'],
        value='Price (Low to High)',
        description='Sort by:',
        disabled=False,
        layout=widgets.Layout(width='250px')
    )
    
    apply_button = widgets.Button(
        description='Apply Filters',
        button_style='success',
        icon='filter',
        layout=widgets.Layout(width='150px')
    )
    
    reset_button = widgets.Button(
        description='Reset Filters',
        button_style='warning',
        icon='refresh',
        layout=widgets.Layout(width='150px')
    )
    
    filter_output = widgets.Output()
    
    # Create the layout
    filter_ui = widgets.VBox([
        widgets.HTML("<h3>Advanced Filtering Options</h3>"),
        widgets.HBox([transport_type, departure_time]),
        widgets.HBox([price_range, comfort_level]),
        widgets.HBox([sort_by, apply_button, reset_button]),
        filter_output
    ])
    
    # Define button actions
    def on_apply_button_clicked(b):
        with filter_output:
            clear_output()
            print("Filters applied! This would filter the actual recommendation results.")
            print(f"Transport types: {transport_type.value}")
            print(f"Price range: ₹{price_range.value[0]} - ₹{price_range.value[1]}")
            print(f"Departure times: {departure_time.value}")
            print(f"Comfort level: {comfort_level.value}")
            print(f"Sorting by: {sort_by.value}")
    
    def on_reset_button_clicked(b):
        transport_type.value = ['All']
        price_range.value = [500, 5000]
        departure_time.value = ['All']
        comfort_level.value = ['All']
        sort_by.value = 'Price (Low to High)'
        with filter_output:
            clear_output()
            print("All filters have been reset to default values.")
    
    apply_button.on_click(on_apply_button_clicked)
    reset_button.on_click(on_reset_button_clicked)
    
    return filter_ui

def show_price_trends(source, destination):
    """Display price trend analysis for the selected route"""
    # Generate sample data for demonstration
    dates = pd.date_range(start='2023-10-01', end='2023-10-30', freq='D')
    
    # Create random price data with some patterns
    np.random.seed(42)  # For reproducible results
    base_price = 2000
    
    # Flight prices with weekend spikes
    flight_prices = base_price * 2 + np.random.normal(0, 200, len(dates))
    # Add weekend effect - higher prices on Fri/Sat/Sun
    for i, date in enumerate(dates):
        if date.dayofweek >= 4:  # Fri, Sat, Sun
            flight_prices[i] += 500
    
    # Train prices with slight variations
    train_prices = base_price * 0.8 + np.random.normal(0, 100, len(dates))
    
    # Bus prices, more stable
    bus_prices = base_price * 0.5 + np.random.normal(0, 50, len(dates))
    
    # Create DataFrame
    price_data = pd.DataFrame({
        'Date': dates,
        'Flight': flight_prices,
        'Train': train_prices,
        'Bus': bus_prices
    })
    
    # Calculate statistics
    stats = {
        'Flight': {
            'min': price_data['Flight'].min(),
            'max': price_data['Flight'].max(),
            'mean': price_data['Flight'].mean(),
            'best_date': price_data.loc[price_data['Flight'].idxmin(), 'Date'].strftime('%Y-%m-%d')
        },
        'Train': {
            'min': price_data['Train'].min(),
            'max': price_data['Train'].max(),
            'mean': price_data['Train'].mean(),
            'best_date': price_data.loc[price_data['Train'].idxmin(), 'Date'].strftime('%Y-%m-%d')
        },
        'Bus': {
            'min': price_data['Bus'].min(),
            'max': price_data['Bus'].max(),
            'mean': price_data['Bus'].mean(),
            'best_date': price_data.loc[price_data['Bus'].idxmin(), 'Date'].strftime('%Y-%m-%d')
        }
    }
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(price_data['Date'], price_data['Flight'], label='Flight', marker='o', linestyle='-', color='blue', alpha=0.7)
    ax.plot(price_data['Date'], price_data['Train'], label='Train', marker='s', linestyle='-', color='red', alpha=0.7)
    ax.plot(price_data['Date'], price_data['Bus'], label='Bus', marker='^', linestyle='-', color='green', alpha=0.7)
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Price (₹)')
    ax.set_title(f'Price Trends: {source} to {destination}')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Highlight weekends with light shading
    for i, date in enumerate(dates):
        if date.dayofweek >= 5:  # Sat, Sun
            ax.axvspan(date - pd.Timedelta('0.5D'), date + pd.Timedelta('0.5D'), alpha=0.1, color='gray')
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Create a summary widget
    summary_html = f"""
    <h3>Price Analysis Summary</h3>
    <table style="width:100%; border-collapse: collapse;">
        <tr style="background-color: #f2f2f2;">
            <th style="padding: 8px; text-align: left; border: 1px solid #ddd;">Transport Mode</th>
            <th style="padding: 8px; text-align: left; border: 1px solid #ddd;">Minimum Price</th>
            <th style="padding: 8px; text-align: left; border: 1px solid #ddd;">Maximum Price</th>
            <th style="padding: 8px; text-align: left; border: 1px solid #ddd;">Average Price</th>
            <th style="padding: 8px; text-align: left; border: 1px solid #ddd;">Best Date to Travel</th>
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd;">Flight</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Flight']['min']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Flight']['max']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Flight']['mean']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">{stats['Flight']['best_date']}</td>
        </tr>
        <tr style="background-color: #f9f9f9;">
            <td style="padding: 8px; border: 1px solid #ddd;">Train</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Train']['min']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Train']['max']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Train']['mean']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">{stats['Train']['best_date']}</td>
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd;">Bus</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Bus']['min']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Bus']['max']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">₹{stats['Bus']['mean']:.2f}</td>
            <td style="padding: 8px; border: 1px solid #ddd;">{stats['Bus']['best_date']}</td>
        </tr>
    </table>
    
    <div style="margin-top: 15px; padding: 10px; background-color: #e8f4f8; border-left: 5px solid #2196F3; border-radius: 4px;">
        <p><strong>💡 Travel Insight:</strong> Prices tend to be higher on weekends. Consider traveling mid-week for the best deals.</p>
    </div>
    """
    
    return fig, widgets.HTML(summary_html)

# Example usage
# filter_ui = create_filter_widgets()
# display(filter_ui)
# fig, summary = show_price_trends("Delhi", "Mumbai")
# display(fig)
# display(summary)

# 🌦️ Weather & 🏛️ Local Attractions Features

These cells add weather forecasts for travel dates and local attraction suggestions to help users plan their trips more effectively.

In [ ]:
import random
import datetime
import requests
from IPython.display import Image, display, HTML

def get_weather_forecast(city, days=5):
    """Get weather forecast for destination city (simulated data for demo)"""
    weather_types = ["Sunny", "Partly Cloudy", "Cloudy", "Light Rain", "Heavy Rain", "Thunderstorm", "Windy"]
    weather_icons = {
        "Sunny": "☀️", 
        "Partly Cloudy": "⛅", 
        "Cloudy": "☁️", 
        "Light Rain": "🌦️", 
        "Heavy Rain": "🌧️", 
        "Thunderstorm": "⛈️", 
        "Windy": "💨"
    }
    
    # City temperature ranges (min, max) in Celsius
    city_temps = {
        "Delhi": (15, 35),
        "Mumbai": (24, 33),
        "Bangalore": (18, 28),
        "Hyderabad": (20, 32),
        "Chennai": (24, 35),
        "Kolkata": (22, 33),
        "Pune": (18, 30),
        "Jaipur": (18, 32),
        "Ahmedabad": (22, 36),
        "Agra": (15, 35)
    }
    
    # Use default temperature range if city not in dictionary
    temp_range = city_temps.get(city, (20, 30))
    
    # Generate forecast for the specified number of days
    forecast = []
    today = datetime.datetime.now()
    
    for i in range(days):
        date = today + datetime.timedelta(days=i)
        weather = random.choice(weather_types)
        temp_min = random.randint(temp_range[0], temp_range[0] + 5)
        temp_max = random.randint(temp_range[1] - 5, temp_range[1])
        
        forecast.append({
            'date': date.strftime('%Y-%m-%d'),
            'day': date.strftime('%A'),
            'weather': weather,
            'weather_icon': weather_icons[weather],
            'temp_min': temp_min,
            'temp_max': temp_max,
            'humidity': random.randint(40, 90),
            'wind': random.randint(5, 25)
        })
    
    return forecast

def display_weather_forecast(city, forecast):
    """Display the weather forecast in a nice format"""
    html = f"""
    <div style="max-width: 600px; margin: 0 auto;">
        <h3>🌦️ Weather Forecast for {city}</h3>
        <div style="display: flex; flex-wrap: wrap; justify-content: space-between;">
    """
    
    for day in forecast:
        html += f"""
        <div style="flex-basis: 19%; min-width: 100px; border: 1px solid #ddd; border-radius: 8px; margin-bottom: 10px; padding: 10px; text-align: center; background-color: #f9f9f9;">
            <div style="font-weight: bold;">{day['day']}</div>
            <div style="color: #666;">{day['date']}</div>
            <div style="font-size: 28px; margin: 10px 0;">{day['weather_icon']}</div>
            <div>{day['weather']}</div>
            <div style="margin-top: 5px;">
                <span style="color: #d35400;">{day['temp_max']}°C</span> / 
                <span style="color: #2980b9;">{day['temp_min']}°C</span>
            </div>
            <div style="font-size: 12px; color: #666;">
                Humidity: {day['humidity']}%<br>
                Wind: {day['wind']} km/h
            </div>
        </div>
        """
    
    html += """
        </div>
        <div style="margin-top: 10px; padding: 10px; background-color: #e8f4f8; border-radius: 4px; font-size: 14px;">
            <strong>Note:</strong> Weather data is simulated for demonstration purposes. Check actual weather forecasts before traveling.
        </div>
    </div>
    """
    
    return HTML(html)

def get_local_attractions(city):
    """Get local attraction suggestions for the destination city"""
    attractions = {
        "Delhi": [
            {"name": "Red Fort", "type": "Historical", "rating": 4.5, "description": "UNESCO World Heritage site and iconic symbol of India's heritage", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/9/96/Red_Fort_in_Delhi_03-2016_img3.jpg/800px-Red_Fort_in_Delhi_03-2016_img3.jpg"},
            {"name": "Qutub Minar", "type": "Historical", "rating": 4.4, "description": "UNESCO site with the world's tallest brick minaret", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/09/Qutb_Minar_wide_lense.jpg/800px-Qutb_Minar_wide_lense.jpg"},
            {"name": "India Gate", "type": "Monument", "rating": 4.6, "description": "War memorial honoring Indian soldiers who died in WWI", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c0/India_Gate_in_New_Delhi_03-2016_img3.jpg/800px-India_Gate_in_New_Delhi_03-2016_img3.jpg"},
        ],
        "Mumbai": [
            {"name": "Gateway of India", "type": "Monument", "rating": 4.5, "description": "Historic monument overlooking the Arabian Sea", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Mumbai_Aug_2018_%2843397784544%29.jpg/800px-Mumbai_Aug_2018_%2843397784544%29.jpg"},
            {"name": "Marine Drive", "type": "Scenic", "rating": 4.7, "description": "Iconic seaside promenade known as the 'Queen's Necklace'", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dc/Marine_Drive_Mumbai_Evening.jpg/800px-Marine_Drive_Mumbai_Evening.jpg"},
            {"name": "Elephanta Caves", "type": "Historical", "rating": 4.3, "description": "Ancient cave temples dedicated to Lord Shiva", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8f/Elephanta_Caves_Trimurti.jpg/800px-Elephanta_Caves_Trimurti.jpg"},
        ],
        "Bangalore": [
            {"name": "Lalbagh Botanical Garden", "type": "Nature", "rating": 4.4, "description": "Historic garden with diverse plant species", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/Lal_Bagh%2C_Bangalore%2C_India_%28April%29.jpg/800px-Lal_Bagh%2C_Bangalore%2C_India_%28April%29.jpg"},
            {"name": "Bangalore Palace", "type": "Historical", "rating": 4.2, "description": "Tudor-style palace with beautiful architecture", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Front_view_of_Bangalore_Palace.jpg/800px-Front_view_of_Bangalore_Palace.jpg"},
            {"name": "Cubbon Park", "type": "Nature", "rating": 4.5, "description": "Historic park with lush greenery in the heart of the city", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/41/Cubbon_Park_Monday_View.JPG/800px-Cubbon_Park_Monday_View.JPG"},
        ]
    }
    
    # Default attractions for cities not in the list
    default_attractions = [
        {"name": "City Center", "type": "Urban", "rating": 4.0, "description": "Explore the heart of the city", "image_url": "https://via.placeholder.com/300x200?text=City+Center"},
        {"name": "Local Museum", "type": "Cultural", "rating": 4.1, "description": "Learn about the local history and culture", "image_url": "https://via.placeholder.com/300x200?text=Local+Museum"},
        {"name": "City Park", "type": "Nature", "rating": 4.2, "description": "Relax amid beautiful natural surroundings", "image_url": "https://via.placeholder.com/300x200?text=City+Park"},
    ]
    
    return attractions.get(city, default_attractions)

def display_attractions(city, attractions):
    """Display local attractions in a nice format"""
    html = f"""
    <div style="max-width: 800px; margin: 0 auto;">
        <h3>🏛️ Top Attractions in {city}</h3>
        <div style="display: flex; flex-wrap: wrap; justify-content: space-between;">
    """
    
    for attraction in attractions:
        html += f"""
        <div style="flex-basis: 31%; min-width: 250px; border: 1px solid #ddd; border-radius: 8px; margin-bottom: 15px; background-color: white; box-shadow: 0 2px 5px rgba(0,0,0,0.1); overflow: hidden;">
            <div style="height: 150px; overflow: hidden;">
                <img src="{attraction['image_url']}" style="width: 100%; height: 100%; object-fit: cover;">
            </div>
            <div style="padding: 15px;">
                <div style="font-weight: bold; font-size: 18px;">{attraction['name']}</div>
                <div style="margin: 5px 0;">
                    <span style="background-color: #f0f0f0; padding: 3px 8px; border-radius: 12px; font-size: 12px;">{attraction['type']}</span>
                    <span style="float: right; color: #f39c12;">{'★' * int(attraction['rating'])}{'☆' * (5 - int(attraction['rating']))}</span>
                </div>
                <p style="color: #666; font-size: 14px; margin-top: 10px;">{attraction['description']}</p>
            </div>
        </div>
        """
    
    html += """
        </div>
        <div style="margin-top: 10px; padding: 10px; background-color: #f9e8f5; border-radius: 4px; font-size: 14px;">
            <strong>💡 Travel Tip:</strong> Consider purchasing attraction tickets online in advance to avoid long queues and get better prices.
        </div>
    </div>
    """
    
    return HTML(html)

def travel_extras(city):
    """Show weather and attractions for a city"""
    # Get weather forecast
    forecast = get_weather_forecast(city)
    weather_display = display_weather_forecast(city, forecast)
    
    # Get local attractions
    attractions = get_local_attractions(city)
    attractions_display = display_attractions(city, attractions)
    
    return weather_display, attractions_display

# Example usage
# weather_display, attractions_display = travel_extras("Delhi")
# display(weather_display)
# display(attractions_display)